# Text Topic Extraction

### RevRoad `Dataset` Locations:

- https://huggingface.co/RevolutionCrossroads
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810 (247 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/loc_chronicling_america_1770-1810_issues (707 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files (9.4 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/nara_revolutionary_war_pension_files_PDFs (4.23 TB)
- https://huggingface.co/datasets/RevolutionCrossroads/si_us_revolutionary_era_collections (7.7 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_OCR_results/viewer/block_level (19.5 GB)
- https://huggingface.co/datasets/RevolutionCrossroads/RevX_NER_Entity_Occurrences (939 MB)

### Install and Import Libraries

In [ ]:
# Install BERTopic
# Everything else is already installed in Google Colab
!pip install bertopic

In [ ]:
import numpy as np
import spacy

from datasets import load_dataset

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

from sentence_transformers import SentenceTransformer
from umap import UMAP

### Load and Filter Text Dataset

In [ ]:
# Set up dataset stream
ocr_ds = load_dataset("RevolutionCrossroads/RevX_OCR_results",
                      name="block_level",
                      split="train",
                      streaming=True)
ocr_ds.features

In [ ]:
# For filtering data stream
def is_pension(example):
  return example["collection"] == "rev_war_pension"

def is_news(example):
  return example["collection"] == "chronicling_america"

def is_news_block(example):
  return (
    example["collection"] == "chronicling_america" and
    example["block_text"]and
    len(example["block_text"]) > 100
  )

In [ ]:
# Filter stream and get N blocks of text
n_news = 2 ** 14 # 16k
news_ds = ocr_ds.filter(is_news_block)
news_blocks_ds = news_ds.take(n_news)
news_blocks_text = list(news_blocks_ds["block_text"])
len(news_blocks_text)

In [ ]:
# Preprocessing to remove stopwords and punctuation
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

# Custom stopwords
custom_stopwords = set(ENGLISH_STOP_WORDS).union({
  "company",
  "really",
  "also",
  "would",
  "shall",
  "only",
  "ditto",
  "thy",
  "your"
})

def preprocess(text):
  doc = nlp(text.lower())

  return " ".join([
    token.lemma_
    for token in doc
    if not token.is_stop
    and not token.is_punct
    and not token.is_space
    and token not in custom_stopwords
    and len(token) > 2
    and token.is_alpha
  ])

# Pre-process text
docs = [preprocess(doc) for doc in news_blocks_text]

## LDA Topic Extraction

A more classic way of extracting topics.

Gives us the words that make up the extracted topics.

In [ ]:
# Convert documents into document-term matrix
vectorizer = CountVectorizer(
  lowercase=True,
  stop_words=list(custom_stopwords),
  max_df=0.8,
  min_df=5,
  max_features=10000,
  ngram_range=(1, 2),
  token_pattern=r"(?u)\b[a-zA-Z]{3,}\b"
)
docs_vct = vectorizer.fit_transform(docs)

# Vectorizer vocab
feature_names = vectorizer.get_feature_names_out()

In [ ]:
# Use term counts for LDA
n_topics = 10

lda = LatentDirichletAllocation(
  n_components=n_topics,
  max_iter=10,
  learning_method="online",
  learning_offset=1.0,
  random_state=1010,
).fit(docs_vct)

In [ ]:
# Print words that make up the extracted topics
n_words=16

for topic_idx, topic in enumerate(lda.components_):
  top_indices = (-topic).argsort()[:n_words]
  words = [feature_names[i] for i in top_indices]
  print(f"Topic {topic_idx}: {', '.join(words)}")

## BERTopic

ML embedding topic extraction.

Similar to LDA, but using a model that extracts semantic information instead of word counts.

In [ ]:
# Define all of the components needed by the BERTopic methodology

# Turns text into numbers
sentence_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")

# Helps find some similarities between docs
umap_model = UMAP(
  n_neighbors=50, n_components=5,
  metric="cosine", min_dist=0.1,
  random_state=1010, transform_seed=1010,
)

# Groups docs into clusters
n_topics = 10
cluster_model = KMeans(n_clusters=n_topics, random_state=101010)

# Extracts representative words from the groups
representation_model = KeyBERTInspired(top_n_words=32)

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=cluster_model,
  representation_model=representation_model,
  language="english",
  top_n_words=32
)

In [ ]:
# Extract topics using semantic similarity
topics, probs = topic_model.fit_transform(docs)
_, topic_cnts = np.unique(topic_model.topics_, return_counts=True)

n_words = 16
for topic_idx, topic in enumerate(topic_model.get_topics().values()):
  words = [w for w,c in topic[:n_words]]
  print(f"Topic {topic_idx} ({topic_cnts[topic_idx]}): {', '.join(words)}")